In [ ]:
%load_ext autoreload
%autoreload 2
import dataclasses

from datetime import date
from dateutil.relativedelta import relativedelta
from decimal import Decimal
from moneypy.securities import import_isos_from_yaml
from moneypy.apps.equity_tool import (
    run_scenarios,
    visualize_scenario,
    ExerciseStrategy,
    ISOScenario,
)
from moneypy.tax import Income, RegularTaxSystem, AlternativeMinimumTaxSystem
from plotly.subplots import make_subplots

rts = RegularTaxSystem()
amt = AlternativeMinimumTaxSystem()

base_scenario = ISOScenario(
    num_to_exercise=0,
    num_to_sell=0,
    fair_market_value=Decimal(10.94),
    price_at_exercise=Decimal(28.19),
    price_at_sale=Decimal(50.00),
    exercise_strategy=ExerciseStrategy.INCREASING_STRIKE,
    exercise_date=date.today(),
    sale_date=date.today() + relativedelta(years=1),
    tax_system=rts,
)

delta = 2500

equity_path = "/home/jon/workspaces/finance/moneypy/equity.yaml"
with open(equity_path) as equity_file:
    isos = import_isos_from_yaml(equity_file)

isos = [dataclasses.replace(iso, exercise_date=None, sale_date=None) for iso in isos]

share_count = sum([iso.num_shares for iso in isos])

scenarios = [
    dataclasses.replace(
        base_scenario, num_to_sell=num_to_sell, num_to_exercise=num_to_exercise
    )
    for num_to_exercise in range(0, share_count + delta, delta)
    for num_to_sell in range(0, num_to_exercise + delta, delta)
]

scenarios += [dataclasses.replace(scenario, tax_system=amt) for scenario in scenarios]
income = Income(Decimal(240_000))
df = run_scenarios(
    isos=isos,
    income=income,
    scenarios=scenarios,
)

keys = list(dataclasses.asdict(base_scenario).keys()) + ["year"]
keys.remove("tax_system")

max_tax_df = df.loc[df.groupby(keys)["tax"].idxmax()].reset_index(drop=True)
parameters = ["tax", "cash_flow"]

summary_cols = ["iso_exercise_cost", "iso_proceeds", "cash_flow", "iso_realized_gain", "tax"]
keys.remove("year")
summary_df = (
    max_tax_df.groupby(keys).agg({**{col: "sum" for col in summary_cols}}).reset_index()
)

x_col = "num_to_exercise"
y_col = "num_to_sell"

for parameter in summary_cols:
    fig = make_subplots(cols=1 + len(max_tax_df["year"].unique()))
    row = 1
    col = 1
    for group, group_df in max_tax_df.groupby("year"):
        for trace in visualize_scenario(
            group_df[x_col], group_df[y_col], group_df[parameter]
        ):
            fig.add_trace(trace, row=row, col=col)
        fig.update_xaxes(title_text="Exercise Count", row=1, col=col)
        fig.update_yaxes(title_text="Sell Count", row=1, col=col)
        col += 1

    for trace in visualize_scenario(
        summary_df[x_col], summary_df[y_col], summary_df[parameter]
    ):
        fig.add_trace(trace, row=1, col=col)
    fig.update_xaxes(title_text="Exercise Count", row=1, col=col)
    fig.update_yaxes(title_text="Sell Count", row=1, col=col)
    fig.update_layout(title={"text": parameter.replace("_", " ").title(), "x": 0.5}, height=500, width=1500)
    
    fig.show()
